U2 AquaVigia: modelo de series de tiempo (criterio 7)

Dimension U2 de Jhenderson Machaca: pH y turbidez en tiempo real.

Se entrena con el HISTORIAL REAL del USGS un modelo que proyecta cada variable a UNA HORA (12 pasos de 5 min), se valida contra dos referencias (repetir el ultimo valor y predecir la media) y se guarda el ganador para aplicarlo despues en streaming (cuaderno C).
Tambien se calculan los umbrales de advertencia por estacion y la incertidumbre del modelo, que sirven para estimar el riesgo de alerta.

Antes de correr: copiar `historial.jsonl` (generado con `usgs_productor.py historial`) a la carpeta `data` de esta sesion (`s12-u2-usgs\data\historial.jsonl`).

In [1]:
import os, json, time, math
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

INTEGRANTE = "Jhenderson Machaca"
VARIABLES = ["ph","turbidez_fnu"]

BASE = "/opt/s12-u2-usgs"
ARTIFACTS = f"{BASE}/artifacts"
DATA = f"{BASE}/data/historial.jsonl"
RUTA_MODELOS = f"{ARTIFACTS}/modelos_jhenderson"
os.makedirs(RUTA_MODELOS, exist_ok=True)
if not os.path.exists(DATA):
    raise FileNotFoundError(f"Falta {DATA}. Copia historial.jsonl a la carpeta data de esta sesion.")

spark = (SparkSession.builder.appName("u2-usgs-modelo").master("local[*]")
         .config("spark.sql.shuffle.partitions", "8").config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

esquema_evento = (StructType().add("evento_id", StringType()).add("estacion_id", StringType())
    .add("variable", StringType()).add("codigo_parametro", StringType()).add("valor", DoubleType())
    .add("fecha_medicion_utc", StringType()).add("fecha_ingesta_utc", StringType())
    .add("retraso_min", DoubleType()).add("origen", StringType()).add("fuente", StringType()))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 13:50:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Historial real: que hay

Lecturas del USGS por estacion y variable. Solo se usan las variables de esta dimension.

In [2]:
h = (spark.read.schema(esquema_evento).json(DATA)
     .where(F.col("variable").isin(VARIABLES) & F.col("valor").isNotNull())
     .withColumn("ts", F.to_timestamp("fecha_medicion_utc")))
print("lecturas del historial (mis variables):", h.count())
resumen = (h.groupBy("estacion_id", "variable").agg(F.count("*").alias("lecturas"), F.min("ts").alias("desde"),
           F.max("ts").alias("hasta"), F.round(F.avg("valor"), 2).alias("promedio")))
resumen.orderBy("estacion_id", "variable").show(60, truncate=False)

lecturas del historial (mis variables): 663774


[Stage 3:=============================>                            (8 + 8) / 16]

+---------------+------------+--------+-------------------+-------------------+--------+
|estacion_id    |variable    |lecturas|desde              |hasta              |promedio|
+---------------+------------+--------+-------------------+-------------------+--------+
|USGS-01491000  |ph          |25886   |2026-08-01 00:40:00|2026-10-29 23:30:00|7.26    |
|USGS-01491000  |turbidez_fnu|25840   |2026-08-01 00:40:00|2026-10-29 23:30:00|4.7     |
|USGS-01579550  |ph          |25880   |2026-08-01 00:40:00|2026-10-30 00:05:00|7.89    |
|USGS-01579550  |turbidez_fnu|24580   |2026-08-01 00:40:00|2026-10-28 07:10:00|3.82    |
|USGS-01589290  |ph          |25617   |2026-08-01 00:40:00|2026-10-29 23:50:00|7.66    |
|USGS-01589290  |turbidez_fnu|25554   |2026-08-01 00:40:00|2026-10-29 23:50:00|3.46    |
|USGS-01594441  |ph          |25864   |2026-08-01 00:40:00|2026-10-30 00:05:00|7.54    |
|USGS-01594441  |turbidez_fnu|25815   |2026-08-01 00:40:00|2026-10-30 00:05:00|17.14   |
|USGS-01640152  |ph  

## 2. Serie regular de 5 minutos y tabla ancha

Se redondea cada lectura a su bloque de 5 minutos y se deja una columna por variable. Solo se conservan estaciones que tienen TODAS las variables de la dimension.

In [3]:
PASO = 300
base = (h.withColumn("ts5", (F.floor(F.col("ts").cast("long") / PASO) * PASO).cast("long"))
        .groupBy("estacion_id", "variable", "ts5").agg(F.avg("valor").alias("valor")))
ancho = (base.groupBy("estacion_id", "ts5").pivot("variable", VARIABLES).agg(F.first("valor"))
         .na.drop(subset=VARIABLES).cache())
print("filas (estacion x 5 min con todas mis variables):", ancho.count())
print("estaciones utilizables:", ancho.select("estacion_id").distinct().count())
ancho.orderBy("estacion_id", "ts5").show(5)

filas (estacion x 5 min con todas mis variables): 328019
estaciones utilizables: 13
+-------------+----------+---+------------+
|  estacion_id|       ts5| ph|turbidez_fnu|
+-------------+----------+---+------------+
|USGS-01491000|1785562800|7.0|         5.3|
|USGS-01491000|1785563100|7.1|         5.1|
|USGS-01491000|1785563400|7.1|         5.2|
|USGS-01491000|1785563700|7.1|         5.1|
|USGS-01491000|1785564000|7.1|         5.1|
+-------------+----------+---+------------+
only showing top 5 rows


## 3. Variables del modelo

Para proyectar una variable `v` una hora adelante se usan: su valor actual, sus rezagos (5, 10, 15, 30 y 60 min), el cambio de la ultima hora, el valor actual de las otras variables de la dimension y la hora del dia (seno y coseno).
Solo se admiten filas con los rezagos y la meta EXACTAMENTE a los tiempos esperados (sin huecos).

In [4]:
LAGS = [1, 2, 3, 6, 12]
HORIZONTE = 12      # 12 pasos de 5 min = 1 hora

def columnas(v):
    return [v] + [f"{v}_l{k}" for k in LAGS] + ["delta_1h"] + [u for u in VARIABLES if u != v] + ["hora_sin", "hora_cos"]

def construir(v):
    w = Window.partitionBy("estacion_id").orderBy("ts5")
    d = (ancho.withColumn("ts5_meta", F.lead("ts5", HORIZONTE).over(w))
              .withColumn("label", F.lead(v, HORIZONTE).over(w)))
    cond = (F.col("ts5_meta") - F.col("ts5") == HORIZONTE * PASO)
    for k in LAGS:
        d = d.withColumn(f"{v}_l{k}", F.lag(v, k).over(w)).withColumn(f"_t{k}", F.lag("ts5", k).over(w))
        cond = cond & (F.col("ts5") - F.col(f"_t{k}") == k * PASO)
    d = (d.where(cond)
          .withColumn("delta_1h", F.col(v) - F.col(f"{v}_l12"))
          .withColumn("hora", (F.col("ts5") % 86400) / 3600.0)
          .withColumn("hora_sin", F.sin(F.col("hora") * (2 * math.pi / 24)))
          .withColumn("hora_cos", F.cos(F.col("hora") * (2 * math.pi / 24))))
    return d.select("estacion_id", "ts5", "label", *columnas(v)).na.drop()

CORTE = ancho.approxQuantile("ts5", [0.8], 0.001)[0]
print("Corte cronologico (80 % del tiempo):", pd.to_datetime(CORTE, unit="s", utc=True))

[Stage 36:>                                                         (0 + 8) / 8]

Corte cronologico (80 % del tiempo): 2026-10-12 01:55:00+00:00


## 4. Entrenamiento, validacion y eleccion del ganador

Division cronologica (el entrenamiento solo usa lo que termina antes del corte; la prueba, lo posterior). Se comparan dos modelos contra dos referencias con RMSE, MAE y R2.

In [5]:
ev = {m: RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}
def evaluar(df, nombre):
    r = {"Modelo": nombre, "RMSE": ev["rmse"].evaluate(df), "MAE": ev["mae"].evaluate(df), "R2": ev["r2"].evaluate(df)}
    print(f"  {nombre}: RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}  R2={r['R2']:.4f}")
    return r

TABLAS, META = {}, {"variables": {}, "corte": int(CORTE), "horizonte_min": HORIZONTE * 5, "integrante": INTEGRANTE}
for v in VARIABLES:
    print(f"\n===== {v} =====")
    ds = construir(v).cache()
    train = ds.where(F.col("ts5") + HORIZONTE * PASO <= CORTE)
    test = ds.where(F.col("ts5") > CORTE)
    n_tr, n_te = train.count(), test.count()
    print(f"  filas entrenamiento={n_tr}  prueba={n_te}")
    if n_tr < 500 or n_te < 100:
        print("  POCOS DATOS: baja la exigencia o baja mas historial (--dias 90)."); continue
    cols = columnas(v)
    media = train.agg(F.avg("label")).first()[0]
    filas = [evaluar(test.withColumn("prediction", F.col(v)), "Referencia: repetir el ultimo valor"),
             evaluar(test.withColumn("prediction", F.lit(media)), "Referencia: media de entrenamiento")]
    modelos = {}
    ens = lambda: VectorAssembler(inputCols=cols, outputCol="features", handleInvalid="skip")
    cand = {"LinearRegression": LinearRegression(featuresCol="features", labelCol="label", regParam=0.01),
            "RandomForest": RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=40, maxDepth=7, seed=42)}
    for nombre, est in cand.items():
        m = Pipeline(stages=[ens(), est]).fit(train)
        modelos[nombre] = m
        filas.append(evaluar(m.transform(test), nombre))
    tabla = pd.DataFrame(filas)
    ganador = tabla[tabla["Modelo"].isin(cand)].sort_values("RMSE").iloc[0]
    mejor = modelos[ganador["Modelo"]]
    mejor.write().overwrite().save(f"{RUTA_MODELOS}/modelo_{v}_1h")
    rm_p = tabla.loc[0, "RMSE"]; rm_m = tabla.loc[1, "RMSE"]
    META["variables"][v] = {"features": cols, "sigma": float(ganador["RMSE"]), "ganador": ganador["Modelo"],
        "rmse": float(ganador["RMSE"]), "mae": float(ganador["MAE"]), "r2": float(ganador["R2"]),
        "rmse_persistencia": float(rm_p), "rmse_media": float(rm_m),
        "mejora_vs_persistencia_pct": float(100 * (rm_p - ganador["RMSE"]) / rm_p)}
    TABLAS[v] = tabla
for v, t in TABLAS.items():
    print(f"\n--- {v} ---"); print(t.round(4).to_string(index=False))


===== ph =====


  filas entrenamiento=250477  prueba=64451
  Referencia: repetir el ultimo valor: RMSE=0.0753  MAE=0.0278  R2=0.9810
  Referencia: media de entrenamiento: RMSE=0.5515  MAE=0.4048  R2=-0.0155


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


  LinearRegression: RMSE=0.0683  MAE=0.0375  R2=0.9844


  RandomForest: RMSE=0.0855  MAE=0.0430  R2=0.9756



===== turbidez_fnu =====


  filas entrenamiento=250477  prueba=64451
  Referencia: repetir el ultimo valor: RMSE=8.5651  MAE=0.9464  R2=0.6973
  Referencia: media de entrenamiento: RMSE=15.5956  MAE=6.0472  R2=-0.0037


  LinearRegression: RMSE=8.2011  MAE=1.1755  R2=0.7225


  RandomForest: RMSE=13.3286  MAE=1.9243  R2=0.2669



--- ph ---
                             Modelo   RMSE    MAE      R2
Referencia: repetir el ultimo valor 0.0753 0.0278  0.9810
 Referencia: media de entrenamiento 0.5515 0.4048 -0.0155
                   LinearRegression 0.0683 0.0375  0.9844
                       RandomForest 0.0855 0.0430  0.9756

--- turbidez_fnu ---
                             Modelo    RMSE    MAE      R2
Referencia: repetir el ultimo valor  8.5651 0.9464  0.6973
 Referencia: media de entrenamiento 15.5956 6.0472 -0.0037
                   LinearRegression  8.2011 1.1755  0.7225
                       RandomForest 13.3286 1.9243  0.2669


## 5. Lectura honesta del resultado

Predecir una hora adelante en agua es dificil de mejorar: "repetir el ultimo valor" suele ser una referencia fuerte. Si el modelo no la supera, se dice asi.

In [6]:
for v, m in META["variables"].items():
    mej = m["mejora_vs_persistencia_pct"]
    if mej <= 0:
        lectura = "NO supera a repetir el ultimo valor."
    elif mej < 5:
        lectura = "mejora marginal (<5 %) frente a repetir el ultimo valor."
    else:
        lectura = "mejora material frente a repetir el ultimo valor."
    print(f"{v}: ganador={m['ganador']}  RMSE={m['rmse']:.4f}  vs persistencia={m['rmse_persistencia']:.4f}  ({mej:+.1f} %)  -> {lectura}")

ph: ganador=LinearRegression  RMSE=0.0683  vs persistencia=0.0753  (+9.4 %)  -> mejora material frente a repetir el ultimo valor.
turbidez_fnu: ganador=LinearRegression  RMSE=8.2011  vs persistencia=8.5651  (+4.2 %)  -> mejora marginal (<5 %) frente a repetir el ultimo valor.


## 6. Umbrales de advertencia por estacion

Cada estacion tiene su propia historia: la advertencia se define con el percentil 5 y 95 de lo observado en la parte de ENTRENAMIENTO. La norma (critico) se declara aparte en el cuaderno C.

In [7]:
umb = (base.where(F.col("ts5") <= CORTE)
       .groupBy("estacion_id", "variable")
       .agg(F.percentile_approx("valor", 0.05).alias("p05"), F.percentile_approx("valor", 0.95).alias("p95"),
            F.count("*").alias("n")))
umb.orderBy("estacion_id", "variable").show(40, truncate=False)
META["umbrales"] = {f"{r['estacion_id']}|{r['variable']}": {"p05": r["p05"], "p95": r["p95"], "n": r["n"]} for r in umb.collect()}

+---------------+------------+---+----+-----+
|estacion_id    |variable    |p05|p95 |n    |
+---------------+------------+---+----+-----+
|USGS-01491000  |ph          |7.0|7.5 |20683|
|USGS-01491000  |turbidez_fnu|3.6|7.1 |20643|
|USGS-01579550  |ph          |7.4|8.5 |20658|
|USGS-01579550  |turbidez_fnu|0.9|7.8 |19854|
|USGS-01589290  |ph          |7.5|7.8 |20409|
|USGS-01589290  |turbidez_fnu|1.2|9.7 |20347|
|USGS-01594441  |ph          |7.2|7.7 |20642|
|USGS-01594441  |turbidez_fnu|5.4|53.8|20594|
|USGS-01640152  |ph          |7.5|7.9 |20656|
|USGS-01640152  |turbidez_fnu|3.7|33.3|17925|
|USGS-0164578734|ph          |6.9|7.6 |20677|
|USGS-0164578734|turbidez_fnu|1.0|20.4|20664|
|USGS-0164579522|ph          |6.0|6.6 |20171|
|USGS-0164579522|turbidez_fnu|1.6|24.8|20133|
|USGS-01646500  |ph          |8.2|8.8 |20679|
|USGS-01646500  |turbidez_fnu|2.4|10.8|20643|
|USGS-01649190  |ph          |7.3|7.6 |20444|
|USGS-01649190  |turbidez_fnu|0.3|8.2 |20446|
|USGS-01649500  |ph          |7.4|

In [8]:
with open(f"{RUTA_MODELOS}/meta.json", "w", encoding="utf-8") as f:
    json.dump(META, f, ensure_ascii=False, indent=1)
print("Modelos y metadatos guardados en:", RUTA_MODELOS)
print(os.listdir(RUTA_MODELOS))

Modelos y metadatos guardados en: /opt/s12-u2-usgs/artifacts/modelos_jhenderson
['meta.json', 'modelo_ph_1h', 'modelo_turbidez_fnu_1h']
